# 01 — Download GSE176078 (Wu et al. 2021)

Downloads the processed count matrix + metadata from GEO.

**Do NOT commit the downloaded files to GitHub.** They are excluded via `.gitignore`. The archive is ~500 MB.

**Output**: extracted files in `data/raw/Wu_etal_2021_BRCA_scRNASeq/`

In [ ]:
# Add project src/ to path so we can import the config module
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from src import config as cfg

print(f'Project root: {cfg.PROJECT_ROOT}')
print(f'Raw dir:      {cfg.RAW_DIR}')
print(f'DEV_MODE:     {cfg.DEV_MODE}')

In [ ]:
# Download the tar archive from GEO if not already present
import urllib.request
import ssl
from pathlib import Path

tar_path = cfg.RAW_DIR / cfg.GSE_TAR_FILENAME

if tar_path.exists():
    print(f'Already downloaded: {tar_path} ({tar_path.stat().st_size / 1e6:.1f} MB)')
else:
    print(f'Downloading from {cfg.GSE_TAR_URL}...')
    # NCBI occasionally has cert issues; only disable if needed
    ctx = ssl.create_default_context()
    with urllib.request.urlopen(cfg.GSE_TAR_URL, context=ctx) as resp, open(tar_path, 'wb') as f:
        f.write(resp.read())
    print(f'Downloaded: {tar_path} ({tar_path.stat().st_size / 1e6:.1f} MB)')

In [ ]:
# Extract the archive
import tarfile

expected_dir = cfg.RAW_DIR / 'Wu_etal_2021_BRCA_scRNASeq'

if expected_dir.exists() and any(expected_dir.iterdir()):
    print(f'Already extracted: {expected_dir}')
    for f in sorted(expected_dir.iterdir()):
        print(f'  {f.name}')
else:
    print(f'Extracting {tar_path}...')
    with tarfile.open(tar_path, 'r:gz') as tar:
        tar.extractall(cfg.RAW_DIR)
    print(f'Extracted to {expected_dir}')
    for f in sorted(expected_dir.iterdir()):
        print(f'  {f.name}')

## Load into AnnData

The Wu et al. archive contains:

- `count_matrix_sparse.mtx` — sparse count matrix (genes × cells, Matrix Market format)
- `count_matrix_genes.tsv` — gene symbols
- `count_matrix_barcodes.tsv` — cell barcodes
- `metadata.csv` — per-cell metadata: Patient ID, subtype, authors' cell type annotations, etc.

Note: in some versions of this archive the mtx is stored as genes × cells. Scanpy's `read_mtx` gives cells × genes by default, so we may need to transpose. We check dimensions against `metadata.csv` to know which orientation we have.

In [ ]:
import pandas as pd
import scanpy as sc
import anndata as ad

sc.settings.verbosity = 2

# Load metadata first to know how many cells we expect
meta = pd.read_csv(cfg.RAW_METADATA, index_col=0)
print(f'Metadata: {meta.shape[0]} cells, {meta.shape[1]} columns')
print(f'Columns: {list(meta.columns)}')
meta.head()

In [ ]:
# Report key annotation columns
print(f'--- {cfg.BATCH_KEY} ---')
print(meta[cfg.BATCH_KEY].value_counts().head(30))
print(f'\nTotal patients: {meta[cfg.BATCH_KEY].nunique()}')

if cfg.SUBTYPE_COLUMN in meta.columns:
    print(f'\n--- {cfg.SUBTYPE_COLUMN} ---')
    print(meta[cfg.SUBTYPE_COLUMN].value_counts())

if cfg.CELLTYPE_COLUMN_AUTHORS in meta.columns:
    print(f'\n--- {cfg.CELLTYPE_COLUMN_AUTHORS} ---')
    print(meta[cfg.CELLTYPE_COLUMN_AUTHORS].value_counts())

In [ ]:
# Load the count matrix. scanpy.read_mtx returns cells-by-genes if the mtx is
# stored that way, otherwise genes-by-cells. We'll check orientation by size.
import scipy.io
import scipy.sparse as sp

genes = pd.read_csv(cfg.RAW_GENES, header=None, sep='\t')[0].values
barcodes = pd.read_csv(cfg.RAW_BARCODES, header=None, sep='\t')[0].values
print(f'{len(genes)} genes, {len(barcodes)} barcodes')

mat = scipy.io.mmread(cfg.RAW_MATRIX).tocsr()
print(f'Matrix shape (as loaded): {mat.shape}')

# Wu et al. store genes as rows, cells as columns. We want cells as rows
# for AnnData. Transpose if needed.
if mat.shape == (len(genes), len(barcodes)):
    print('Transposing from (genes, cells) -> (cells, genes)')
    mat = mat.T.tocsr()
elif mat.shape == (len(barcodes), len(genes)):
    print('Already (cells, genes)')
else:
    raise ValueError(f'Matrix shape {mat.shape} does not match '
                     f'genes ({len(genes)}) or barcodes ({len(barcodes)})')

In [ ]:
# Build the AnnData object
adata = ad.AnnData(
    X=mat,
    obs=pd.DataFrame(index=barcodes),
    var=pd.DataFrame(index=genes),
)
adata.var_names_make_unique()

# Attach per-cell metadata. Align by barcode.
common = adata.obs_names.intersection(meta.index)
print(f'Barcodes in both matrix and metadata: {len(common)}')
adata = adata[common].copy()
adata.obs = meta.loc[common].copy()

print(f'Final AnnData: {adata.n_obs} cells x {adata.n_vars} genes')
print(f'ARC in dataset: {"ARC" in adata.var_names}')
adata

In [ ]:
# Save raw AnnData. We write in compressed h5ad to keep disk usage down.
print(f'Writing {cfg.H5AD_RAW}...')
adata.write_h5ad(cfg.H5AD_RAW, compression='gzip')
print(f'Done. File size: {cfg.H5AD_RAW.stat().st_size / 1e6:.1f} MB')

---

**Next**: `02_qc_filtering.ipynb`